# Local Wakeword Bootstrap Notebook

This notebook is the notebook-first entrypoint for local development.
It runs environment checks and phase commands from inside Jupyter.

> Current repository status: `cli` phase commands are stubs. This notebook verifies execution flow first.

In [ ]:
from __future__ import annotations

import os
import platform
import subprocess
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find project root with pyproject.toml")


PROJECT_ROOT = find_project_root(Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
os.chdir(PROJECT_ROOT)


def run(cmd: list[str]) -> int:
    print("$", " ".join(cmd))
    proc = subprocess.run(cmd, cwd=PROJECT_ROOT, check=False)
    print("exit code:", proc.returncode)
    return proc.returncode


In [ ]:
print("Python:", sys.version)
print("Platform:", platform.platform())

major, minor = sys.version_info[:2]
if not (major == 3 and 10 <= minor <= 12):
    print("[warning] Recommended Python version is 3.10~3.12 for TensorFlow compatibility.")
else:
    print("[ok] Python version looks compatible.")

In [ ]:
# Create baseline directories for local runs.
required_dirs = [
    PROJECT_ROOT / "data" / "raw" / "positive",
    PROJECT_ROOT / "data" / "raw" / "negative",
    PROJECT_ROOT / "data" / "synth" / "positive",
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "data" / "manifests",
    PROJECT_ROOT / "models" / "release",
]

for d in required_dirs:
    d.mkdir(parents=True, exist_ok=True)
    print("ensured:", d.relative_to(PROJECT_ROOT))

## Phase Command Smoke Run

Run each phase command from the notebook.
At this point, these commands are expected to print stub output.

In [ ]:
phase_commands = [
    [sys.executable, "-m", "nubjuk_wakeword.cli", "check-env"],
    [sys.executable, "-m", "nubjuk_wakeword.cli", "synth"],
    [sys.executable, "-m", "nubjuk_wakeword.cli", "augment"],
    [sys.executable, "-m", "nubjuk_wakeword.cli", "train"],
    [sys.executable, "-m", "nubjuk_wakeword.cli", "eval"],
    [sys.executable, "-m", "nubjuk_wakeword.cli", "export"],
    [sys.executable, "-m", "nubjuk_wakeword.cli", "release"],
]

results = {}
for cmd in phase_commands:
    name = cmd[-1]
    results[name] = run(cmd)

print("\nSummary:")
for name, code in results.items():
    print(f"- {name}: {code}")

## Next Step

Implement the command internals in `src/nubjuk_wakeword/cli.py` and split logic into phase modules (`data`, `train`, `eval`, `export`).
As internals are implemented, this notebook becomes your reproducible local runbook.